In [0]:
# %pip install sentence-transformers pandas
import pandas as pd
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [0]:
df = spark.read.table("samplesuperstore.bronzedata.orders")
df = df.toPandas()

df['Productname'] = df['Productname'].str.lower().str.strip()
df['Category'] = df['Category'].str.lower()
df['Subcategory'] = df['Subcategory'].str.lower()
df['Segment'] = df['Segment'].str.lower()

In [0]:
%sql
SELECT current_schema();


In [0]:
from sentence_transformers import SentenceTransformer
import pandas as pd

texts = df['Productname'].tolist()
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts)

emb_df = pd.DataFrame(embeddings)
spark_emb = spark.createDataFrame(emb_df)
spark_emb.write.mode("overwrite").saveAsTable("samplesuperstore.silverdata.rag_embeddings")


In [0]:
from sklearn.metrics.pairwise import cosine_similarity

query = "Why are furniture profits low?"
query_emb = model.encode([query])

scores = cosine_similarity(query_emb, embeddings)
top_idx = scores[0].argsort()[-5:][::-1]

context = df.iloc[top_idx]
print(context[['Category','Subcategory','Discount','Profit']])


In [0]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = f"Given this data: {context.to_dict()}, explain why profits are low in Furniture."
print(generator(prompt, max_length=80))


In [0]:
insights = pd.DataFrame([{
    "Category":"Furniture",
    "Insight":"High discounts in Furniture reduce profit"
}])
spark.createDataFrame(insights).write.mode("append").saveAsTable("samplesuperstore.silverdata.rag_insights")
